# Lesson 19 · Autoencoders & VAEs: compress digits into a few numbers, then generate new digits from them

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/19-autoencoder-vae-en.ipynb)

This notebook accompanies [Lesson 19 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/19-autoencoder-vae.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We write PCA, an autoencoder and a VAE ourselves in numpy, on real MNIST images, then ask two questions:
**how much is lost when a 784-pixel image is compressed to k numbers**, and **where do we take a code from to
generate a new digit?**

**What you'll do:**
1. Compute PCA from scratch, measure the reconstruction error as a function of k, and see a linear autoencoder learn
   that very same subspace.
2. Write the backpropagation of an autoencoder and of a VAE by hand (with the reparameterization trick), check it
   with finite differences, then train them with a 2-dimensional code.
3. Decode the demo's 2-D latent map, sample from $\mathcal N(0, I)$, from a Gaussian fitted to the codes and from a
   Gaussian mixture, and see where the samples fall into gaps.
4. Measure the variance of the reparameterization trick against the score-function estimator (REINFORCE) on that
   same decoder.
5. Read the results of the β sweep: where the blur comes from, when a VAE collapses, and why Stable Diffusion uses
   an autoencoder only to compress.

**Background:** [Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/),
[Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/) (binary
cross-entropy) and [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/);
the Adam optimizer of [Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/). **Run time:** under
a minute on a CPU, including the first download of MNIST (~11 MB) and of the demo's 1.7 MB of results.

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The full training
runs (20,000 steps × 3 seeds for each configuration, about 45 minutes on 16 cores), however, are loaded from the very
files the demo uses; the notebook reruns a small version of each mechanism and checks what can be recomputed.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label):
# from section 5 on, autoencoder = orange, squares; VAE = blue, round dots
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def span(values, digits=2):
    """The min–max range across seeds, as in the demo: 27.31–27.71."""
    lo, hi = min(values), max(values)
    return num(lo, digits) if round(lo, digits) == round(hi, digits) else f"{num(lo, digits)}–{num(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 1.9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{num(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def show_digits(axes, images, titles=None, fontsize=9):
    """Draw each 784-pixel image (values 0–1) in its own panel: white background, black strokes."""
    for i, ax in enumerate(np.ravel(axes)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(np.asarray(images[i]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=fontsize)


def tile(images, rows, cols, gap=2):
    """Join rows × cols 28×28 images into one large image (much faster than one axis per image)."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527), span([27.307, 27.714]), sci(1.9e-7))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, 28×28 grayscale handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo's runs, we split them into:
- **training:** the first 55,000 images (PCA and the networks learn on this part);
- **validation:** the last 5,000 images of the training file (the demo chooses the learning rate $\eta$ here);
- **test:** the 10,000 images of the test file, used only to report results.

The cell also loads four results files that the demo uses (`fetch_json`, with an MD5 check):
- `vae-compress.json`: PCA, the linear autoencoder and the deep autoencoder with k = 1–64, 3 seeds;
- `vae-latent.json`: the 2-D VAEs and autoencoders, the β sweep, the Gaussian decoder, the gradient estimators;
- `vae-2d-ae.json`, `vae-2d-vae.json`: the two 2-D decoders that the demo runs in the browser (seed 0).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Results of the long training runs (the same files as the demo on the web). build.py replaces COURSE_DATA_URL with
# the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
comp = fetch_json("vae-compress.json", "964972ff4498d4d13d8d72710c2a8789")
lat = fetch_json("vae-latent.json", "67bbda5976a13be0e72011a223954aeb")
packed = {
    "ae": fetch_json("vae-2d-ae.json", "e292fd127fe3e6c7bff5937e3aad1d66"),
    "vae": fetch_json("vae-2d-vae.json", "ef1ececca80f225edaa8b94ce4c836b8"),
}
size = sum(
    (Path("data") / f).stat().st_size
    for f in ["vae-compress.json", "vae-latent.json", "vae-2d-ae.json", "vae-2d-vae.json"]
)
print("training:", train_img.shape, " test:", test_img.shape, f" · the demo's 4 files: {num(size / 1e6, 1)} MB")

Grayscale images have pixels in $[0, 1]$. The VAE's Bernoulli decoder (section 5) models **binary** images: each
pixel is 0 or 1. As in the demo's runs, we binarize stochastically: a pixel of brightness $p$ becomes 1 with
probability $p$. The training images are binarized afresh at every step; the test images are binarized **once**,
with seed 2, so every number on the test set is on the same set of 0/1 images. The cell below checks that the 20
images the demo displays have exactly these bits.

In [ ]:
X_all = train_img.astype(np.float32) / 255
Xtr, Xval = X_all[:55000], X_all[55000:]  # training · validation
Xte, yte = test_img.astype(np.float32) / 255, test_lab.astype(np.int64)  # test
Xte_bin = (np.random.default_rng(2).random((10000, 784), dtype=np.float32) < Xte).astype(np.float32)


def b64(s, dtype):
    """Decode an array that the demo stores as base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


SHOW = lat["digits"]["idx"]  # the demo's 20 test images: the first two images of each digit
assert np.array_equal(b64(lat["digits"]["bits"], np.uint8).reshape(-1, 784), Xte_bin[SHOW])
print(f"training {num(len(Xtr), 0)} · validation {num(len(Xval), 0)} · test {num(len(Xte), 0)} images")
print("digits of the 20 demo images:", [int(yte[i]) for i in SHOW])
print(f"{pct(Xte_bin.mean())} of the binary pixels are 1; mean brightness of the grayscale images {pct(Xte.mean())}")

fig, axes = plt.subplots(2, 10, figsize=(10, 2.3))
show_digits(axes[0], Xte[SHOW[::2]], [f"gray: {yte[i]}" for i in SHOW[::2]])
show_digits(axes[1], Xte_bin[SHOW[::2]], ["binary"] * 10)
plt.show()

The top row shows grayscale images, the bottom row the same images after binarization: the strokes are still
readable, but their edges are jagged, because the faint gray pixels have randomly become 0 or 1.

## 3 · Linear compression: PCA from scratch

An image is a row vector $x$ of 784 numbers. Compressing it linearly to $k$ numbers means choosing $k$ directions
$b_1, \dots, b_k$ (the rows of the matrix $B_k$, of size $k \times 784$, orthonormal), taking the image's coordinates
along those directions, then rebuilding it:

$$z = (x - m)\,B_k^{\top} \quad (k \text{ numbers}), \qquad \hat x = m + z\,B_k, \qquad
\text{error} = \lVert \hat x - x \rVert^2 = \sum_{i=1}^{784} (\hat x_i - x_i)^2$$

where $m$ is the mean image. **PCA** chooses the $k$ directions with the smallest total squared error on the
training set: they are the $k$ eigenvectors with the largest eigenvalues of the covariance matrix $C$ (784 × 784),
and the eigenvalue $\lambda_j$ is the variance of the data along direction $j$. The measure throughout the lesson is
the error above, computed on grayscale images and averaged over the 10,000 test images, as in Hinton &
Salakhutdinov (2006).

There are two ways to compute the same directions: the singular value decomposition (SVD) of the mean-subtracted
data matrix, or the eigendecomposition of $C$. The cell below checks that they agree on 2,000 images, then uses the
second one for the 55,000 images: the SVD of a 55,000 × 784 matrix is much slower, while $C$ is only 784 × 784.

In [ ]:
KS = [1, 2, 4, 8, 16, 30, 64]


def scatter(X):
    """Number of images, sum and the matrix XᵀX: enough to compute the mean and the covariance, and they can be added
    up across sets."""
    X = X.astype(np.float64)
    return len(X), X.sum(0), X.T @ X


def pca_from(n, s, G):
    """Mean, eigenvalues (descending) and principal components (rows of B) from scatter()."""
    m = s / n
    C = (G - n * np.outer(m, m)) / (n - 1)  # 784 × 784 covariance
    lam, V = np.linalg.eigh(C)  # eigh returns the eigenvalues in ascending order
    return m, lam[::-1], V[:, ::-1].T


def pca_sse(m, B, X, k):
    """Reconstruction error (sum of squares over the 784 pixels) of each image, with the first k components."""
    Xc = X.astype(np.float64) - m
    R = (Xc @ B[:k].T) @ B[:k]
    return ((R - Xc) ** 2).sum(1)


# Two ways, one result (on the first 2,000 images)
S = Xtr[:2000].astype(np.float64)
_, sv, Vt = np.linalg.svd(S - S.mean(0), full_matrices=False)
_, lam_s, B_s = pca_from(*scatter(Xtr[:2000]))
cos10 = np.abs((Vt[:10] * B_s[:10]).sum(1))  # cos between the two j-th directions (the sign is arbitrary)
assert np.allclose(sv[:10] ** 2 / 1999, lam_s[:10]) and np.allclose(cos10, 1)
print(
    f"SVD and eigenvalues of C on {num(len(S), 0)} images: the first 10 directions agree (smallest |cos| {num(cos10.min(), 6)})"
)

n55, s55, G55 = scatter(Xtr)  # reused in the next cell
m_pca, lam, B = pca_from(n55, s55, G55)
sse_pca = {k: pca_sse(m_pca, B, Xte, k).mean() for k in KS}
for k in KS:
    share = lam[:k].sum() / lam.sum()
    print(f"k = {k:>2}: error {num(sse_pca[k], 2):>6} per image · keeps {pct(share, 1)} of the variance")
    assert abs(sse_pca[k] - comp["pca"]["sse"][k - 1]) < 2e-3, "differs from the demo: check the data again"
    assert abs(share - comp["pca"]["explained"][k - 1]) < 1e-3
print(f"no compression (only the mean image m): error {num(((Xte - m_pca) ** 2).sum(1).mean(), 2)}")

With $k = 30$, PCA is left with an error of 13.86 per image. Hinton & Salakhutdinov (2006) report 13.87 for
30-dimensional PCA; they fitted PCA on all 60,000 training images. Adding the 5,000 validation images to `scatter`
reproduces exactly that number:

In [ ]:
n5, s5, G5 = scatter(Xval)
m60, _, B60 = pca_from(n55 + n5, s55 + s5, G55 + G5)
sse60 = pca_sse(m60, B60, Xte, 30).mean()
print(f"PCA fitted on 60,000 images, k = 30: {num(sse60, 4)} → rounded {num(sse60, 2)} (paper: 13.87)")
assert round(sse60, 2) == comp["hinton"]["pca"] == 13.87

The figure below compares the original images (top row) with the PCA reconstructions from $k$ = 2, 8 and 30
numbers. Two numbers give only blurry smudges, and many of them look like a different digit (the 7 becomes a stroke
that looks like a 9); 8 numbers already make most of the digits recognizable; 30 numbers give readable digits, but
still smeared, with a gray halo around the strokes.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(10, 4.4))
Xs = Xte[SHOW[::2]].astype(np.float64)
show_digits(axes[0], Xs)
for row, k in zip(axes[1:], [2, 8, 30]):
    show_digits(row, np.clip(m_pca + ((Xs - m_pca) @ B[:k].T) @ B[:k], 0, 1))
for row, label in zip(axes[:, 0], ["original", "PCA k = 2", "PCA k = 8", "PCA k = 30"]):
    row.text(-0.12, 0.5, label, transform=row.transAxes, ha="right", va="center", fontsize=9)
plt.subplots_adjust(left=0.1, wspace=0.05, hspace=0.08)
plt.show()

## 4 · What does a linear autoencoder learn?

An **autoencoder** consists of an encoder $x \mapsto z$ and a decoder $z \mapsto \hat x$, learned together by
gradient descent to minimize the reconstruction error. The bottleneck $z$ has only $k$ numbers, so the network has
to choose what to keep. The simplest case is when both halves are linear:

$$z = (x - m)\,W_e, \qquad \hat x = m + z\,W_d, \qquad W_e: 784 \times k,\;\; W_d: k \times 784$$

Baldi & Hornik (1989) proved that the minimum of this squared error is the PCA subspace. We try it with $k = 16$:
random initialization, Adam (Lesson 07), batch size 128, 1,500 steps, warmup then a cosine schedule.

### 🤔 Predict first
After training, (a) how does the error of the linear autoencoder compare with PCA's at $k = 16$? (b) Does each row
of $W_d$ coincide with a principal component (|cos| ≈ 1)?

In [ ]:
def adam_step(p, g, state, lr, b1=0.9, b2=0.999, eps=1e-8):
    """Adam (Lesson 07), updates p in place; state holds m, v and the step count t."""
    state["t"] = t = state.get("t", 0) + 1
    for k in p:
        m = state.setdefault("m" + k, np.zeros_like(p[k]))
        v = state.setdefault("v" + k, np.zeros_like(p[k]))
        m += (1 - b1) * (g[k] - m)
        v += (1 - b2) * (g[k] ** 2 - v)
        p[k] -= (lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps)).astype(p[k].dtype)


def lr_at(s, lr, steps):
    """Linear warmup over the first 2% of the steps, then cosine down to 0 (as in the demo's runs)."""
    warm = max(1, steps // 50)
    return lr * (s + 1) / warm if s < warm else lr * 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm)))


def train_linear_ae(k, steps=1500, lr=0.01, seed=0, batch=128):
    rng = np.random.default_rng(seed)
    m = Xtr.mean(0)
    p = {
        "We": (rng.standard_normal((784, k)) * np.sqrt(1 / 784)).astype(np.float32),
        "Wd": (rng.standard_normal((k, 784)) * np.sqrt(1 / k)).astype(np.float32),
    }
    state = {}
    for s in range(steps):
        xc = Xtr[rng.integers(0, len(Xtr), batch)] - m
        z = xc @ p["We"]
        r = z @ p["Wd"] - xc  # reconstruction error of each pixel
        dr = 2 * r / batch  # ∂(mean of Σ r²)/∂r
        adam_step(p, {"Wd": z.T @ dr, "We": xc.T @ (dr @ p["Wd"].T)}, state, lr_at(s, lr, steps))
    return m, p


m_lin, p_lin = train_linear_ae(16)
Tc = Xte.astype(np.float64) - m_lin
sse_lin = ((Tc @ p_lin["We"] @ p_lin["Wd"] - Tc) ** 2).sum(1).mean()
print(f"k = 16, {num(1500, 0)} steps: linear autoencoder {num(sse_lin, 3)} · PCA {num(sse_pca[16], 3)}")
# (b) compare each row of W_d with the principal component closest to it
rows = p_lin["Wd"].astype(np.float64)
rows /= np.linalg.norm(rows, axis=1, keepdims=True)
best_cos = np.abs(rows @ B[:16].T).max(1)
print(f"|cos| between each row of W_d and the closest principal component: {num(best_cos.min())}–{num(best_cos.max())}")
# (c) compare the whole subspaces: cosines of the principal angles between span(W_d) and span(first 16 components)
Q = np.linalg.qr(rows.T)[0]
cos_sub = np.linalg.svd(B[:16] @ Q, compute_uv=False)
print(
    f"principal angles between the two subspaces: smallest cos {num(cos_sub.min(), 4)}; overlap {pct((cos_sub**2).mean(), 1)}"
)
assert abs(sse_lin / sse_pca[16] - 1) < 0.005 and (cos_sub**2).mean() > 0.99 and best_cos.max() < 0.9

(a) The error is almost exactly PCA's, and (c) the subspace spanned by $W_d$ coincides with the subspace of the
first 16 principal components to 99.9%. But (b) no row of $W_d$ coincides with a principal component: a linear
autoencoder learns exactly the PCA **subspace**, not the PCA **axes**. Rotating the directions within that subspace
(and rotating back in the encoder) leaves $\hat x$ unchanged, so the error has no reason to prefer one particular
set of axes.

So to compress better than PCA, you need nonlinear activation functions. The figure below plots the error against
$k$ (log axis) for PCA (computed above), the linear autoencoder and the deep 784-256-128-$k$ autoencoder with a
symmetric decoder, both taken from the demo's runs (20,000 steps, 3 seeds, one point per seed), along with the two
numbers Hinton & Salakhutdinov published at $k = 30$ (hollow diamonds).

In [ ]:
lin = {int(k): [s["test"] for s in r["seeds"]] for k, r in comp["linear"].items()}
deep = {int(k): [s["test"] for s in r["seeds"]] for k, r in comp["deep"].items()}
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(KS, [sse_pca[k] for k in KS], color=BLUE, marker="o", lw=2, label="PCA (computed above)")
for k in KS:
    ax.scatter([k] * 3, lin[k], s=70, facecolor="none", edgecolor=GREEN, lw=1.5, zorder=3)
    ax.scatter([k] * 3, deep[k], s=28, marker="s", color=ORANGE, zorder=3)
ax.scatter([], [], s=70, facecolor="none", edgecolor=GREEN, lw=1.5, label="linear autoencoder (3 seeds)")
ax.scatter([], [], s=28, marker="s", color=ORANGE, label="deep autoencoder 784-256-128-k (3 seeds)")
ax.scatter([30, 30], [comp["hinton"]["pca"], comp["hinton"]["ae"]], s=80, marker="D", facecolor="none",
           edgecolor="black", label="Hinton & Salakhutdinov (2006): PCA, 784-1000-500-250-30")  # fmt: skip
ax.set_xscale("log")
ax.set(xlabel="k (numbers in the code)", ylabel="reconstruction error per test image", ylim=(0, 50))
ax.set_title("Compressing to k numbers: linear only matches PCA, nonlinear is far better")
ax.legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()
for k in (2, 30):
    print(f"k = {k:>2}: PCA {num(sse_pca[k])} · linear {span(lin[k])} · deep {span(deep[k])}")
print(f"Hinton & Salakhutdinov, k = 30: PCA {num(comp['hinton']['pca'])}, deep autoencoder {num(comp['hinton']['ae'])}")

The green circles (linear autoencoder) lie right on the PCA line at every $k$. The orange squares (deep
autoencoder) are far lower: with 2 numbers, the error drops from 43.64 to about 27; with 30 numbers, from 13.86 to
about 3.6. Hinton & Salakhutdinov's network (784-1000-500-250-30) is larger, was pretrained layer by layer with
RBMs, and reaches 3.00.

## 5 · An autoencoder and a VAE, by hand

From here on, we work with binary images and a small network that trains in a few seconds:

$$\text{encoder: } h = \mathrm{ReLU}(xW_1 + b_1),\;\; \mu = hW_\mu + b_\mu,\;\; \log\sigma^2 = hW_\sigma + b_\sigma
\qquad \text{decoder: } g = \mathrm{ReLU}(zW_2 + b_2),\;\; a = gW_3 + b_3,\;\; \hat x = \mathrm{sigmoid}(a)$$

with a hidden layer of 128 neurons on each side (the demo's networks are 784-256-128 and symmetric). The decoder
gives each pixel a probability $\hat x_i$, i.e. a Bernoulli distribution $p(x \mid z)$, so the reconstruction error
is the binary cross-entropy of Lesson 04, summed over the 784 pixels:

$$-\log p(x \mid z) = -\sum_{i=1}^{784} \big[x_i \log \hat x_i + (1 - x_i)\log(1 - \hat x_i)\big]
\qquad \frac{\partial(-\log p)}{\partial a_i} = \hat x_i - x_i$$

- **Autoencoder:** $z = \mu$, and the loss is the reconstruction error (there is no $\log\sigma^2$).
- **VAE:** $z = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal N(0, I)$ (the reparameterization
  trick), and the loss is $\mathcal L = -\log p(x \mid z) + \beta\, D_{\mathrm{KL}}$ with
  $D_{\mathrm{KL}} = \tfrac12 \sum_j (\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2)$.

Because $z$ is a differentiable function of $\mu$, $\sigma$ and a fixed noise $\varepsilon$, the gradient flows
through the sampling step as through any other layer: $\partial z/\partial \mu = 1$ and
$\partial z/\partial \log\sigma^2 = \varepsilon \odot \sigma / 2$. The KL adds $\partial D_{\mathrm{KL}}/\partial\mu =
\mu$ and $\partial D_{\mathrm{KL}}/\partial \log\sigma^2 = (\sigma^2 - 1)/2$. The cell below writes both the forward
and the backward pass; `rec="gauss"` is the Gaussian decoder used in section 11.

In [ ]:
def init_net(d, H=128, vae=True, seed=0, dtype=np.float32):
    """He for the ReLU layers, 1/fan-in for μ and the output layer, 0.1 × that for log σ² (as in the demo); biases
    b = 0."""
    rng = np.random.default_rng(seed)
    shapes = {"1": (784, H, 2 / 784), "mu": (H, d, 1 / H), "2": (d, H, 2 / d), "3": (H, 784, 1 / H)}
    if vae:
        shapes["lv"] = (H, d, 0.01 / H)
    p = {}
    for name, (a, b, var) in shapes.items():
        p["W" + name] = (rng.standard_normal((a, b)) * np.sqrt(var)).astype(dtype)
        p["b" + name] = np.zeros(b, dtype)
    return p


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))  # no overflow for large |a|


def bce(a, x):
    """−log p(x|z) of the Bernoulli decoder with logits a, summed over the pixels (Lesson 04)."""
    return (np.maximum(a, 0) - a * x + np.log1p(np.exp(-np.abs(a)))).sum(1)


def encode(p, x):
    h = np.maximum(x @ p["W1"] + p["b1"], 0)
    return h, h @ p["Wmu"] + p["bmu"], (h @ p["Wlv"] + p["blv"] if "Wlv" in p else None)


def decode_logits(p, z):
    g = np.maximum(z @ p["W2"] + p["b2"], 0)
    return g, g @ p["W3"] + p["b3"]


def loss_grad(p, x, eps, beta=1.0, rec="bern", s2=0.5):
    """Batch means of the reconstruction error and of the KL, along with the gradient of rec + β·KL with respect to
    every parameter."""
    B = len(x)
    h, mu, lv = encode(p, x)
    sd = None if lv is None else np.exp(0.5 * lv)
    z = mu if lv is None else mu + sd * eps  # the reparameterization trick
    g, a = decode_logits(p, z)
    if rec == "bern":
        r, da = bce(a, x), sigmoid(a) - x
    else:  # Gaussian with fixed variance σ²: ‖x̂ − x‖² / (2σ²)
        y = sigmoid(a)
        r, da = ((y - x) ** 2).sum(1) / (2 * s2), (y - x) / s2 * y * (1 - y)
    kl = np.zeros(B) if lv is None else 0.5 * (mu**2 + sd**2 - 1 - lv).sum(1)
    G = {}
    da = da / B
    G["W3"], G["b3"] = g.T @ da, da.sum(0)
    dg = (da @ p["W3"].T) * (g > 0)
    G["W2"], G["b2"] = z.T @ dg, dg.sum(0)
    dz = dg @ p["W2"].T
    dmu = dz if lv is None else dz + beta * mu / B
    dh = dmu @ p["Wmu"].T
    if lv is not None:
        dlv = dz * eps * sd / 2 + beta * (sd**2 - 1) / 2 / B
        G["Wlv"], G["blv"] = h.T @ dlv, dlv.sum(0)
        dh += dlv @ p["Wlv"].T
    G["Wmu"], G["bmu"] = h.T @ dmu, dmu.sum(0)
    dh *= h > 0
    G["W1"], G["b1"] = x.T @ dh, dh.sum(0)
    return float(r.mean()), float(kl.mean()), G

A handwritten backward pass easily gets a factor of $\tfrac12$ or a sign wrong. As in Lesson 06, we compare every
gradient with central finite differences $\big(\mathcal L(\theta + h) - \mathcal L(\theta - h)\big) / 2h$ on a small
network in float64 (d = 3, a hidden layer of 8 neurons, 4 images), with $\varepsilon$ held fixed: the
reparameterization trick turns $\mathcal L$ into a deterministic function of the parameters. The cell below prints
the largest relative error for the autoencoder, the Bernoulli VAE and the Gaussian VAE.

In [ ]:
def gradcheck(vae, rec, beta=0.7, seed=0, h=1e-5):
    rng = np.random.default_rng(seed)
    p = init_net(3, H=8, vae=vae, seed=seed, dtype=np.float64)
    for k in p:
        p[k] += 0.1 * rng.standard_normal(p[k].shape)  # so that the biases b are nonzero and some ReLU neurons are off
    x = rng.random((4, 784)) if rec == "gauss" else (rng.random((4, 784)) < 0.3).astype(float)
    eps = rng.standard_normal((4, 3))

    def loss():
        r, kl, _ = loss_grad(p, x, eps, beta, rec, s2=0.3)
        return r + beta * kl

    G = loss_grad(p, x, eps, beta, rec, s2=0.3)[2]
    worst = 0.0
    for k in p:
        flat = p[k].reshape(-1)  # a view: changing flat changes p[k]
        for j in rng.choice(flat.size, min(6, flat.size), replace=False):
            old = flat[j]
            flat[j] = old + h
            up = loss()
            flat[j] = old - h
            down = loss()
            flat[j] = old
            num_, ana = (up - down) / (2 * h), G[k].reshape(-1)[j]
            worst = max(worst, abs(num_ - ana) / max(1e-8, abs(num_) + abs(ana)))
    return worst


for name, vae, rec in [
    ("autoencoder", False, "bern"),
    ("Bernoulli VAE", True, "bern"),
    ("Gaussian VAE", True, "gauss"),
]:
    err = gradcheck(vae, rec)
    print(f"{name:13s}: largest relative error {sci(err)}")
    assert err < 1e-5

Relative errors of order $10^{-7}$ for every parameter tried: the backward pass is correct. (They can't get much
smaller, because $\mathcal L$ is a sum over 784 pixels, a few hundred nats, so the subtraction
$\mathcal L(\theta + h) - \mathcal L(\theta - h)$ loses a few significant digits; a real bug in the backward pass
gives an error of order 1.) Now for training. The cell below defines `train()` (Adam, batch size 64, warmup +
cosine, a freshly binarized batch of images and a fresh $\varepsilon$ at every step) and `evaluate()` on the
10,000 test images:
- the **reconstruction error** $-\log p(x \mid z)$ (nats, with a fixed $\varepsilon$) and the **KL** (nats);
- **SSE**: decode $z = \mu$ and compare with the *grayscale* image, exactly the measure of sections 3 and 4.

## 6 · Compressing to 2 numbers: an autoencoder and a VAE

We train two identical networks, with a bottleneck $d = 2$ so that the codes can be drawn on the plane: an
autoencoder and a VAE (β = 1). Each network gets 1,200 steps, a few seconds; the demo's networks ran for 20,000
steps.

### 🤔 Predict first
The VAE has to pay the extra KL term, and it reconstructs from a noisy $z$. So when decoding $z = \mu$, are the
VAE's images **blurrier** (a clearly larger SSE) than those of an autoencoder with the same architecture?

In [ ]:
def train(d, vae, steps=1200, lr=3e-3, batch=64, seed=0, beta=1.0, rec="bern", s2=0.5, H=128):
    rng = np.random.default_rng(1000 + seed)  # order of the images, binarization and ε
    p, state, hist = init_net(d, H, vae, seed), {}, []
    for s in range(steps):
        idx = rng.integers(0, len(Xtr), batch)
        x = (rng.random((batch, 784), dtype=np.float32) < Xtr[idx]).astype(np.float32) if rec == "bern" else Xtr[idx]
        eps = rng.standard_normal((batch, d), dtype=np.float32)
        r, kl, G = loss_grad(p, x, eps, beta, rec, s2)
        hist.append((r, kl))
        adam_step(p, G, state, lr_at(s, lr, steps))
    return p, np.array(hist)


def evaluate(p, rec="bern", s2=0.5, seed=3):
    X = Xte_bin if rec == "bern" else Xte
    _, mu, lv = encode(p, X)
    z = mu
    if lv is not None:
        z = mu + np.exp(0.5 * lv) * np.random.default_rng(seed).standard_normal(mu.shape).astype(np.float32)
    a = decode_logits(p, z)[1]
    r = bce(a, X).mean() if rec == "bern" else (((sigmoid(a) - X) ** 2).sum(1) / (2 * s2)).mean()
    unit_kl = np.zeros(mu.shape[1]) if lv is None else (0.5 * (mu**2 + np.exp(lv) - 1 - lv)).mean(0)
    sse = ((sigmoid(decode_logits(p, mu)[1]) - Xte) ** 2).sum(1).mean()
    active = int((mu.astype(np.float64).var(0) > 0.01).sum())  # active dimensions (Burda et al., 2016)
    return {"rec": r, "kl": unit_kl.sum(), "unit_kl": unit_kl, "sse": sse, "mu": mu, "active": active}


nets, res = {}, {}
print(f"each network {num(1200, 0)} steps, batch size 64, η = 0.003")
for name, vae in [("ae", False), ("vae", True)]:
    nets[name], _ = train(2, vae)
    res[name] = evaluate(nets[name])
    r = res[name]
    print(
        f"{name:>3}: SSE {num(r['sse'])} · −log p(x|z) {num(r['rec'], 1)} nats · KL {num(r['kl'])} nats · "
        f"standard deviation of the codes ({num(r['mu'][:, 0].std())}, {num(r['mu'][:, 1].std())}) · "
        f"codes range from {num(r['mu'].min(), 1)} to {num(r['mu'].max(), 1)}"
    )
gap = abs(res["ae"]["sse"] - res["vae"]["sse"]) / res["ae"]["sse"]
print(f"SSE difference: {pct(gap, 1)}")
assert gap < 0.05 and 0.6 < res["vae"]["mu"].std(0).min() and res["vae"]["mu"].std(0).max() < 1.8
assert np.abs(res["ae"]["mu"]).max() > 6

ae2, vae2 = lat["groups"]["ae2"]["seeds"], lat["groups"]["vae2"]["seeds"]
print(
    f"the demo's networks (20,000 steps, 3 seeds): SSE autoencoder {span([s['sse'] for s in ae2])}, "
    f"VAE {span([s['sse'] for s in vae2])}"
)

No. The two networks reconstruct images almost equally well: in this short run they differ by a few percent, and the
demo's 20,000-step networks give an SSE of 27.34–27.65 (autoencoder) and 27.26–27.71 (VAE), within the variation
across seeds. In 2 dimensions, **the blur comes from the bottleneck**: just 2 numbers aren't enough to keep the
particular strokes of each image, and the decoder draws the average of the images whose codes lie close together.
The KL is not the main culprit here.

Where the two networks differ is **where they put the codes**. The figure below plots the codes $\mu$ of the first
5,000 test images (gray dots), each digit written at the median of its codes, along with three dashed circles of
radius 1, 2, 3 for $\mathcal N(0, I)$. The two panels have different scales.

In [ ]:
def latent_map(ax, mu, labels, title):
    ax.scatter(mu[:, 0], mu[:, 1], s=2, color=GREY, alpha=0.35, lw=0)
    for c in range(10):
        med = np.median(mu[labels == c], axis=0)
        ax.text(*med, str(c), ha="center", va="center", fontsize=12, weight="bold",
                bbox={"boxstyle": "circle,pad=0.15", "fc": "white", "ec": "black", "lw": 0.8})  # fmt: skip
    t = np.linspace(0, 2 * np.pi, 200)
    for r in (1, 2, 3):
        ax.plot(r * np.cos(t), r * np.sin(t), color="black", ls="--", lw=0.8)
    ax.set(xlabel="z₁", ylabel="z₂", title=title)
    ax.set_aspect("equal")


fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, name in zip(axes, ["ae", "vae"]):
    latent_map(ax, res[name]["mu"][:5000], yte[:5000], {"ae": "autoencoder", "vae": "VAE"}[name] + " (short run above)")
plt.tight_layout()
fmt_axes(fig)
plt.show()
for name in ["ae", "vae"]:
    inside = (np.linalg.norm(res[name]["mu"], axis=1) < 2).mean()
    print(
        f"{name:>3}: {pct(inside, 1)} of the codes lie inside the circle of radius 2 (for N(0, I): {pct(1 - math.exp(-2), 1)})"
    )

The autoencoder's codes have an arbitrary scale and spread out into long streaks: nothing in its loss says where the
codes should lie. The VAE's codes gather around the origin with a standard deviation close to 1, because the KL
penalizes every $\mu_j$ far from 0 and every $\sigma_j$ different from 1. That is what we need to **generate**
images: take $z$ from $\mathcal N(0, I)$, then decode it.

## 7 · The demo's latent map: decoding, interpolation, sampling

From here on we use the two 2-D decoders that the demo runs in the browser: the same 2 → 128 → 256 → 784
architecture as the full runs (20,000 steps, seed 0). To keep the files small, the hidden layers are stored in
float16 and the output layer's matrix in int8 with a float16 scale factor for each column; the cell below unpacks
them. The file also has the $\mu$ and $\sigma$ of the 20 demo images, and the $\log p(x \mid z = \mu)$ that Python
computed for each image: we recompute that number to check our decoder.

In [ ]:
def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float64).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float64)  # one factor per column
    return b64(t["i8"], np.int8).astype(np.float64).reshape(t["shape"]) * scale


dec = {name: {k: unpack(t) for k, t in packed[name]["dec"].items()} for name in packed}


def decode(m, z):
    """E[x|z] = sigmoid(logit) of the unpacked decoder: 784 probabilities for each z."""
    h1 = np.maximum(np.atleast_2d(z) @ m["d0.W"] + m["d0.b"], 0)
    h2 = np.maximum(h1 @ m["d1.W"] + m["d1.b"], 0)
    return sigmoid(h2 @ m["out.W"] + m["out.b"])


def loglik_grad(m, z, x):
    """log p(x|z) of the Bernoulli decoder for one binary image x and many z, along with its gradient with respect to
    z (backpropagation)."""
    h1 = np.maximum(z @ m["d0.W"] + m["d0.b"], 0)
    h2 = np.maximum(h1 @ m["d1.W"] + m["d1.b"], 0)
    a = h2 @ m["out.W"] + m["out.b"]
    dh2 = ((x - sigmoid(a)) @ m["out.W"].T) * (h2 > 0)  # ∂ log p / ∂a = x − x̂
    dh1 = (dh2 @ m["d1.W"].T) * (h1 > 0)
    return -bce(a, x[None]), dh1 @ m["d0.W"].T


print({k: v.shape for k, v in dec["vae"].items()})
D20 = lat["digits"]
mu20, sd20 = np.array(D20["vae"]["mu"]), np.array(D20["vae"]["sd"])
f_mu = np.array([loglik_grad(dec["vae"], mu20[i][None], Xte_bin[SHOW[i]].astype(np.float64))[0][0] for i in range(20)])
want = np.array([g["fMu"] for g in D20["vae"]["grad"]])
print(
    f"log p(x | z = μ) of the 20 demo images: {num(f_mu.min(), 1)} … {num(f_mu.max(), 1)} nats; largest deviation from Python {sci(np.abs(f_mu - want).max())}"
)
assert np.abs(f_mu - want).max() < 1e-3, "differs from the demo: check the decoder again"

The unpacked decoder gives exactly the $\log p(x \mid z)$ that Python computed when it exported the file (a
deviation under $10^{-3}$ nats). The file also has the codes $\mu$ of the first 5,000 test images under both models.
The figure below decodes a 13 × 13 grid of points $z$, spanning the 1st to the 99th percentile of the codes along
each axis: each tile is the image $\mathbb E[x \mid z]$ at the coordinates marked on the axes.

In [ ]:
codes = {name: b64(lat["map"]["codes"][name], "<f2").astype(np.float64).reshape(-1, 2) for name in ("ae", "vae")}
code_lab = b64(lat["map"]["labels"], np.uint8)
assert np.array_equal(code_lab, yte[:5000])
NAME = {"ae": "autoencoder", "vae": "VAE"}
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, name in zip(axes, ["ae", "vae"]):
    lo, hi = np.quantile(codes[name], [0.01, 0.99], axis=0)
    g1, g2 = np.linspace(lo[0], hi[0], 13), np.linspace(hi[1], lo[1], 13)  # the top row is the largest z₂
    tiles = decode(dec[name], np.array([[a, b] for b in g2 for a in g1])).reshape(13, 13, 28, 28)
    ax.imshow(tiles.transpose(0, 2, 1, 3).reshape(13 * 28, 13 * 28), cmap="gray_r", vmin=0, vmax=1,
              extent=[lo[0], hi[0], lo[1], hi[1]], aspect="auto")  # fmt: skip
    ax.set(xlabel="z₁", ylabel="z₂", title=f"{NAME[name]}: decoding a grid of z")
    sse = ((decode(dec[name], codes[name]) - Xte[:5000]) ** 2).sum(1).mean()
    print(f"{NAME[name]:>11}: 1st–99th percentiles of the codes: z₁ ∈ [{num(lo[0], 1)}, {num(hi[0], 1)}], z₂ ∈ [{num(lo[1], 1)}, {num(hi[1], 1)}]; "
          f"SSE of the 5,000 reconstructed images {num(sse)}")  # fmt: skip
plt.tight_layout()
fmt_axes(fig)
plt.show()
med = np.array([np.median(codes["ae"][code_lab == c], 0) for c in range(10)])  # the median code of each digit
near0 = [c for c in range(10) if np.abs(med[c]).max() < 0.5]
lo, hi = np.quantile(codes["ae"], [0.01, 0.99], axis=0)
step = (hi - lo) / 12  # one step of the 13 × 13 grid on the left
print(f"autoencoder: the median codes of the digits {', '.join(map(str, near0))} are less than 0.5 from the origin "
      f"on each axis; one grid step is {num(step[0])} × {num(step[1])}")  # fmt: skip
assert near0 == [2, 3, 5, 8] and step.min() > 1

Both maps are continuous: take a small step on the plane and the digit gradually changes into the neighboring
digit. The difference is the scale: the autoencoder's codes spread over a region several times wider than the region
$[-3, 3]^2$ where $\mathcal N(0, I)$ usually falls, while the VAE's codes fit neatly inside it. On the VAE's grid,
every digit has its own place. Not on the autoencoder's: the codes of 2, 3, 5 and 8 crowd around the origin (their
medians are less than 0.5 from it on each axis), while one grid step is 1.93 × 1.67, so the grid hardly draws them.

One thing that's easy to forget: the tiles above are **means** $\mathbb E[x \mid z]$, i.e. the probability that
each pixel is 1, not samples from the model. A true sample of $p(x \mid z)$ draws each pixel as 0 or 1 with that
probability:

In [ ]:
z0 = mu20[0]  # the code of the first demo image (a 7) in the VAE
p0 = decode(dec["vae"], z0)[0]
rng = np.random.default_rng(5)
fig, axes = plt.subplots(1, 5, figsize=(7, 1.8))
show_digits(axes, [Xte_bin[SHOW[0]], p0] + [(rng.random(784) < p0).astype(float) for _ in range(3)],
            ["binary image x", "E[x|z]", "sample 1", "sample 2", "sample 3"])  # fmt: skip
plt.show()
print(f"z = μ = ({num(z0[0])}, {num(z0[1])}), σ = ({num(sd20[0][0], 3)}, {num(sd20[0][1], 3)})")

The Bernoulli samples are speckled: each pixel gets its own coin flip, independent of its neighbors. This decoder
can't model the relationships between pixels, so the “smooth image” that most VAE tutorials show is a mean, and part
of its blur comes from the averaging itself.

### Interpolation
Walk in a straight line from the code of one image to the code of another, and decode every point along the way
(top row: autoencoder, bottom row: VAE; from the 7 to the 0 among the 20 demo images):

In [ ]:
mus = {"ae": np.array(D20["ae"]["mu"]), "vae": mu20}
a_i, b_i = 0, 6  # demo image 0 (a 7) and demo image 6 (a 0)
fig, axes = plt.subplots(2, 9, figsize=(10, 2.6))
for row, name in zip(axes, ["ae", "vae"]):
    path = [(1 - t) * mus[name][a_i] + t * mus[name][b_i] for t in np.linspace(0, 1, 9)]
    show_digits(row, decode(dec[name], np.array(path)))
    row[0].text(-0.15, 0.5, NAME[name], transform=row[0].transAxes, ha="right", va="center", fontsize=9)
plt.subplots_adjust(left=0.1, wspace=0.05)
plt.show()
mid = {n: [s["mid"] for s in lat["groups"][n + "2"]["seeds"]] for n in ("ae", "vae")}
rec_fd = [s["recFd"] for n in ("ae", "vae") for s in lat["groups"][n + "2"]["seeds"]]
print(f"FD of the midpoints of 10,000 pairs of images (3 seeds): autoencoder {span(mid['ae'])}, VAE {span(mid['vae'])}; "
      f"reconstructions {span(rec_fd)}; two sets of real images {num(lat['judge']['floor'])}")  # fmt: skip

Both interpolate: the path passes through the digits that lie between the 7 and the 0 on the map. To measure this
with a number, the demo uses **FD** (Fréchet distance): the computation of FID (Heusel et al., 2017), but with the
Inception network replaced by the LeNet of Lesson 12, so FD values can only be compared with each other within the
course. The smaller the FD, the more the set of images resembles real images; two sets of real images give an FD of
1.32. The midpoints of both models look less like real images than the reconstructions do. The VAE is a little
better (the two sets of 3 seeds don't overlap), but the gap is much smaller than the gap between the midpoints and
the reconstructions.

## 8 · Sampling: $\mathcal N(0, I)$, a Gaussian fitted to the codes, or a Gaussian mixture?

To generate a new digit, we take a point $z$ and decode it. The question is which distribution to take $z$ from. We
try three ways on both maps:
- **$\mathcal N(0, I)$**, the VAE's prior;
- **a Gaussian fitted to the codes**: the same mean and covariance as the codes;
- **a mixture of 10 Gaussians (GMM)** fitted to the codes with the EM algorithm, *after* training is over (Ghosh et
  al. call this *ex-post density estimation*).

The cell below writes EM for a GMM with full covariances, initialized with k-means++. Each iteration has two steps:
- **E step:** compute the probability that each component generated each point;
- **M step:** re-estimate the weight, mean and covariance of each component from those probabilities.

We fit on the codes of the first 4,000 test images and hold out the remaining 1,000 to define a **gap**: a sample
falls into a gap if it is farther from the nearest code than 99% of the held-out codes are. We also record the label
of the nearest code, to see whether the samples cover all 10 digits.

In [ ]:
def gmm_fit(Z, K=10, iters=100, seed=0):
    rng = np.random.default_rng(seed)
    N, d = Z.shape
    centers = [Z[rng.integers(N)]]
    for _ in range(K - 1):  # k-means++: pick each new center far from the existing ones
        d2 = np.min(((Z[:, None] - np.array(centers)[None]) ** 2).sum(-1), 1)
        centers.append(Z[rng.choice(N, p=d2 / d2.sum())])
    mu, cov, w = np.array(centers), np.array([np.cov(Z, rowvar=False)] * K), np.full(K, 1 / K)
    for _ in range(iters):
        logp = np.empty((N, K))  # E step: log [w_k · N(z | μ_k, Σ_k)], dropping the shared constant
        for k in range(K):
            L = np.linalg.cholesky(cov[k])
            u = np.linalg.solve(L, (Z - mu[k]).T)
            logp[:, k] = np.log(w[k]) - 0.5 * (u**2).sum(0) - np.log(np.diag(L)).sum()
        R = np.exp(logp - logp.max(1, keepdims=True))
        R /= R.sum(1, keepdims=True)  # responsibilities: component k generated point n with probability R[n, k]
        Nk = R.sum(0) + 1e-10  # M step
        w, mu = Nk / N, (R.T @ Z) / Nk[:, None]
        for k in range(K):
            Dk = Z - mu[k]
            cov[k] = (R[:, k, None] * Dk).T @ Dk / Nk[k] + 1e-6 * np.eye(d)
    return w, mu, cov


def gmm_sample(g, n, rng):
    w, mu, cov = g
    comp = rng.choice(len(w), n, p=w)
    out = np.empty((n, mu.shape[1]))
    for k in range(len(w)):  # each sample sits at its own random place in comp, so the first 24 mix all the components
        out[comp == k] = rng.multivariate_normal(mu[k], cov[k], (comp == k).sum())
    return out


def nearest(Q, R):
    """Distance to the nearest point in R, and its index, for each row of Q."""
    out = [((Q[i : i + 500, None] - R[None]) ** 2).sum(-1) for i in range(0, len(Q), 500)]
    return np.sqrt(np.concatenate([o.min(1) for o in out])), np.concatenate([o.argmin(1) for o in out])


SAMPLERS = {"prior": "N(0, I)", "gauss": "fitted Gaussian", "gmm": "10-component GMM"}
samples, live = {}, {}
for name in ("ae", "vae"):
    ref, held = codes[name][:4000], codes[name][4000:]
    r99 = np.quantile(nearest(held, ref)[0], 0.99)
    print(f"{NAME[name]}: fitted on {num(len(ref), 0)} codes, {num(2000, 0)} samples per method; gap = farther from the nearest code "
          f"than {num(r99, 3)} (99% quantile of the {num(len(held), 0)} held-out codes)")  # fmt: skip
    rng = np.random.default_rng(7)
    samples[name] = {
        "prior": rng.standard_normal((2000, 2)),
        "gauss": rng.multivariate_normal(ref.mean(0), np.cov(ref, rowvar=False), 2000),
        "gmm": gmm_sample(gmm_fit(ref), 2000, rng),
    }
    for k, z in samples[name].items():
        dist, j = nearest(z, ref)
        share = np.bincount(code_lab[j], minlength=10) / len(z)
        live[name, k] = {"hole": (dist > r99).mean(), "rare": share.min(), "rare_cls": int(share.argmin())}
        print(
            f"{NAME[name]:>11}, z ~ {SAMPLERS[k]:<17}: {pct(live[name, k]['hole'], 1):>5} of the samples fall into a gap; "
            f"rarest digit ({live[name, k]['rare_cls']}) makes up {pct(live[name, k]['rare'], 1)}"
        )
assert live["ae", "prior"]["rare"] < 0.01 and live["ae", "gauss"]["hole"] > max(0.03, 3 * live["ae", "prior"]["hole"])
assert live["vae", "prior"]["hole"] < 0.05 and live["vae", "prior"]["rare"] > 0.03

Drawing $z$ from $\mathcal N(0, I)$ for the autoencoder puts almost no samples in a gap, but almost none of them is
a 1: the circle of radius 3 around the origin covers only a small corner of the autoencoder's map. For the VAE,
$\mathcal N(0, I)$ covers all 10 digits. The figure below shows 24 samples from each method (top row: autoencoder;
bottom row: VAE).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.4))
for r, name in enumerate(("ae", "vae")):
    for c, k in enumerate(SAMPLERS):
        axes[r, c].imshow(tile(decode(dec[name], samples[name][k][:24]), 4, 6), cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{NAME[name]}, z ~ {SAMPLERS[k]}", fontsize=10)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

### Is the right scale all it takes?
The fitted Gaussian has exactly the mean and covariance of the autoencoder's codes. So do its samples look much more
like real digits than the samples from $\mathcal N(0, I)$? The cell below prints the FD of 10,000 samples for each
sampling method, from the demo's full runs (3 seeds), along with the FD of the reconstructions.

In [ ]:
G = lat["groups"]


def fd(name, k):
    return [s[k]["fd"] for s in G[name]["seeds"]]


def holes(name, k):
    return [s[k]["hole"] for s in G[name]["seeds"]]


def compare(na, a, nb, b):
    """As in the demo: when the two sets of 3 seeds overlap, we can't yet say which one is better (a smaller FD is
    better)."""
    if min(b) <= max(a) and min(a) <= max(b):
        return "within the variation across seeds"
    return f"{na if max(a) < min(b) else nb} is better"


print(
    "FD of 10,000 samples against the test images (3 seeds, the demo's full runs; two sets of real images give 1.32):"
)
for n in ("ae2", "vae2"):
    row = " · ".join(f"{SAMPLERS[k]} {span(fd(n, k))}" for k in SAMPLERS)
    print(f"  {NAME[n[:-1]]:>11}: reconstructions {span([s['recFd'] for s in G[n]['seeds']])} · {row}")
gap_ae = [100 * h for h in holes("ae2", "gauss")], [100 * h for h in holes("ae2", "prior")]
print(f"autoencoder: {compare('fitted Gaussian', fd('ae2', 'gauss'), 'N(0, I)', fd('ae2', 'prior'))}; samples in a "
      f"gap: {span(gap_ae[0], 1)}% with the fitted Gaussian, {span(gap_ae[1], 1)}% with N(0, I)")  # fmt: skip
print(
    f"autoencoder + GMM vs. VAE + N(0, I): {compare('autoencoder + GMM', fd('ae2', 'gmm'), 'VAE', fd('vae2', 'prior'))}"
)
print(f"VAE: {compare('GMM', fd('vae2', 'gmm'), 'N(0, I)', fd('vae2', 'prior'))}")
GHOSH = {"AE, N(0, I)": 58.73, "AE + GMM": 10.66, "VAE, N(0, I)": 19.21}  # Ghosh et al. (2020), Table 1, MNIST, FID
print("Ghosh et al. (2020), FID on MNIST: " + " · ".join(f"{k} {num(v)}" for k, v in GHOSH.items()))

No. The fitted Gaussian has the right scale for the codes, but the codes aren't shaped like a Gaussian: they lie in
clusters and streaks, with empty regions in between. The Gaussian covers those regions too, where the decoder was
never trained (4.5–7.1% of the samples fall into a gap), so its FD is not clearly better than that of
$\mathcal N(0, I)$: the two sets of 3 seeds overlap.

A mixture of 10 Gaussians, on the other hand, follows the clusters, and the autoencoder + GMM gives an FD on par
with the VAE sampled from $\mathcal N(0, I)$: within the variation across seeds. Ghosh et al. (2020) found the same
thing on MNIST with the real FID: the autoencoder sampled from $\mathcal N(0, I)$ gives 58.73, and adding a GMM
brings it down to 10.66, against 19.21 for the VAE sampled from $\mathcal N(0, I)$ (their Table 1). The VAE's real
advantage is that it **comes with** a distribution to sample from; the autoencoder has to learn that step on top.

Even the VAE samples better when a GMM is fitted to its own codes. Averaged over images, the KL splits into two
parts (Hoffman & Johnson, 2016): the information the code carries about the image, plus the distance from the
**aggregate distribution of the codes** $q(z) = \mathbb E_x\, q(z \mid x)$ to $\mathcal N(0, I)$. The second part
never quite reaches 0, so $\mathcal N(0, I)$ still lands in regions where few codes lie (Dai & Wipf, 2019). With
higher-dimensional codes, that distance is larger:

In [ ]:
for n, label in [("vae2", "d = 2"), ("d16", "d = 16"), ("b1", "d = 50")]:
    print(f"VAE {label:>6}: FD with N(0, I) {span(fd(n, 'prior'))} · with GMM {span(fd(n, 'gmm'))} · "
          f"reconstructions {span([s['recFd'] for s in G[n]['seeds']])}")  # fmt: skip

With $d = 16$, the reconstructions are nearly real (FD about 2), but samples from $\mathcal N(0, I)$ only reach
about 31, while samples from a GMM reach about 8. Most of the distance between the samples and the real images lies
in **where $z$ is taken from**, not in the decoder.

## 9 · The reparameterization trick: why not REINFORCE?

Training a VAE needs the gradient with respect to $\mu$ (and $\sigma$) of $\mathbb E_{z \sim q}[\log p(x \mid z)]$,
an expectation under the very distribution we are learning. There are two ways to estimate it from a sample
$z = \mu + \sigma \odot \varepsilon$:

- **reparameterization:** $\nabla_z \log p(x \mid z)$, backpropagated through the decoder (the function
  `loglik_grad` above);
- **score function** (REINFORCE): $\log p(x \mid z) \cdot \nabla_\mu \log q(z \mid x) =
  \log p(x \mid z)\cdot \varepsilon / \sigma$. It only needs the *value* $\log p(x \mid z)$, so it works even when
  you can't differentiate with respect to $z$ (discrete variables, rewards in reinforcement learning). Subtracting
  a **baseline** $\log p(x \mid z = \mu)$ doesn't change the expectation, because $\mathbb E[\varepsilon] = 0$.

All three are unbiased. We measure them on the decoder of the 2-D VAE, with the $\mu$ and $\sigma$ of the 20 demo
images, 3,000 samples of $\varepsilon$ per image. The demo computed the “true gradient” of each image from 200,000
samples.

### 🤔 Predict first
How many times larger is the variance of one REINFORCE sample (without a baseline) than that of one
reparameterization sample: 2, 10, 100, or more still?

In [ ]:
N_EPS = 3000
est_var, z_worst, err_curve = {"sf": [], "sfb": []}, 0.0, None
for i in range(20):
    x = Xte_bin[SHOW[i]].astype(np.float64)
    eps = np.random.default_rng(300 + i).standard_normal((N_EPS, 2))
    f, g = loglik_grad(dec["vae"], mu20[i] + sd20[i] * eps, x)
    est = {"rp": g, "sf": f[:, None] * eps / sd20[i], "sfb": (f - f_mu[i])[:, None] * eps / sd20[i]}
    var = {k: ((e - e.mean(0)) ** 2).sum(1).mean() for k, e in est.items()}  # trace of the covariance matrix
    true = np.array(D20["vae"]["grad"][i]["true"])
    for k, e in est.items():  # how many standard errors is the mean of each estimator from the true gradient?
        z_worst = max(z_worst, np.max(np.abs(e.mean(0) - true) / (e.std(0) / math.sqrt(N_EPS))))
    for k in ("sf", "sfb"):
        est_var[k].append(var[k] / var["rp"])
    if i == 0:  # error curve as a function of the number of samples, for the figure below
        var0, n = var, np.arange(1, N_EPS + 1)
        err_curve = {k: np.linalg.norm(np.cumsum(e, 0) / n[:, None] - true, axis=1) for k, e in est.items()}
ratio = {k: np.array(v) for k, v in est_var.items()}
ref = {k: np.array([q["var"][k] / q["var"]["rp"] for q in D20["vae"]["grad"]]) for k in ("sf", "sfb")}
print(f"all three estimators match the true gradient: largest deviation {num(z_worst, 1)} standard errors")
for k, label in [("sf", "REINFORCE"), ("sfb", "REINFORCE + baseline")]:
    print(f"{label:>20}: variance {num(ratio[k].min(), 0 if k == 'sf' else 1)}–{num(ratio[k].max(), 0 if k == 'sf' else 1)} "
          f"times larger (median {num(np.median(ratio[k]), 0 if k == 'sf' else 1)}) · demo, 200,000 samples: "
          f"{num(ref[k].min(), 0 if k == 'sf' else 1)}–{num(ref[k].max(), 0 if k == 'sf' else 1)} times")  # fmt: skip
assert z_worst < 5 and np.abs(np.log(ratio["sf"] / ref["sf"])).max() < 0.1
assert np.abs(np.log(ratio["sfb"] / ref["sfb"])).max() < 0.25 and ratio["sf"].min() > 500

Thousands of times larger: 768–49,780 times depending on the image, matching the numbers the demo computed from
200,000 samples. A variance $r$ times larger means you need $r$ REINFORCE samples to be as accurate as **one**
reparameterization sample. The figure below plots, for the first demo image (a 7), the distance from the mean of the
first $n$ samples to the true gradient, as a function of $n$ (both axes logarithmic). The dashed gray line is the
$1/\sqrt n$ slope that every unbiased estimator follows; only the height differs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
n = np.arange(1, N_EPS + 1)
style = {
    "rp": ("reparameterization", BLUE, "-"),
    "sfb": ("REINFORCE + baseline", GREEN, (0, (6, 2))),
    "sf": ("REINFORCE", ORANGE, (0, (1, 1.5))),
}
for k, (label, color, ls) in style.items():
    ax.plot(n, err_curve[k], color=color, ls=ls, lw=2)
    ax.text(N_EPS * 1.08, err_curve[k][-1], label.replace(" + ", " +\n"), color=color, va="center", fontsize=9)
ax.plot(n, err_curve["rp"][9] * np.sqrt(10 / n), color=GREY, ls="--", lw=1, label="slope 1/√n")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(xlim=(1, N_EPS * 15), xlabel="number of samples ε averaged", ylabel="‖mean − true gradient‖")
ax.set_title(
    f"Gradient estimates with respect to μ for one image (true gradient length {num(np.linalg.norm(D20['vae']['grad'][0]['true']), 1)})"
)
ax.legend(frameon=False, fontsize=8, loc="lower left")
fmt_axes(fig)
plt.show()
print(" · ".join(f"{style[k][0]} after {num(N_EPS, 0)} samples: off by {num(err_curve[k][-1], 2)}" for k in style))
print(f"expected error √(variance / n): reparameterization, 1 sample: {num(math.sqrt(var0['rp']), 1)} · REINFORCE, "
      f"{num(N_EPS, 0)} samples: {num(math.sqrt(var0['sf'] / N_EPS), 1)} · log p(x | z = μ) of this image: {num(f_mu[0], 1)} nats")  # fmt: skip

After 3,000 samples, REINFORCE is still off by more than the length of the true gradient itself. In terms of
variance, the mean of 3,000 REINFORCE samples (expected error 34.3) is still worse than a single reparameterization
sample (22.3). The reason: REINFORCE uses only **one number** $\log p(x \mid z)$ (−77.8 nats at
$z = \mu$ for this image) multiplied by $\varepsilon/\sigma$, and this image's $\sigma$ is only (0.092, 0.048), so
each sample is a huge vector that is right only *on average*. Reparameterization uses the derivative
$\partial \log p / \partial z$, which backpropagation provides almost for free. The baseline subtracts most of that
value of about −80, so it helps a lot, but the variance is still 3.6–19.1 times larger.

When each method is actually used for training (the demo's runs, 20,000 steps, 3 seeds), that gap turns into nats:

In [ ]:
print(
    f"{'':>6} {'reparameterization':>18} {'REINFORCE + baseline':>22} {'REINFORCE':>14}   (−ELBO on the test images, nats)"
)
for d, rp in [(2, "vae2"), (16, "d16"), (50, "b1")]:
    cells = [span([-s["elbo"] for s in G[n]["seeds"]], 1) for n in (rp, f"sfb{d}", f"sf{d}")]
    med = {n: np.median([-s["elbo"] for s in G[n]["seeds"]]) for n in (rp, f"sfb{d}", f"sf{d}")}
    print(f"d = {d:>2} {cells[0]:>18} {cells[1]:>22} {cells[2]:>14}   median worse than reparameterization by "
          f"{num(med[f'sfb{d}'] - med[rp], 1)} and {num(med[f'sf{d}'] - med[rp], 1)} nats")  # fmt: skip

With a baseline, REINFORCE still learns a usable VAE, only 3.2–8.6 nats worse (median); without a baseline it is
49.3–66.8 nats worse, and the seeds end up far apart. The reparameterization trick isn't the *only* way to get a
gradient through the sampling step; it is the **low-variance** way. With discrete latent variables you can't write
$z = \mu + \sigma \odot \varepsilon$, so people still use REINFORCE with a baseline, or approximations such as
Gumbel-softmax or straight-through (VQ-VAE).

## 10 · The ELBO is only a lower bound on $\log p(x)$

With β = 1, $-\mathcal L$ is the **ELBO**: $\log p(x) \ge \mathbb E_q[\log p(x \mid z)] - D_{\mathrm{KL}}(q \,\|\, p)$.
The gap is exactly $D_{\mathrm{KL}}\big(q(z \mid x) \,\|\, p(z \mid x)\big)$, how far the cloud $q$ is from the true
posterior. Taking $K$ samples $z_k \sim q$ gives a tighter bound (IWAE, Burda et al., 2016):

$$\log p(x) \;\ge\; \mathbb E\Big[\log \tfrac1K \textstyle\sum_{k=1}^{K} \tfrac{p(x \mid z_k)\, p(z_k)}{q(z_k \mid x)}\Big],$$

a bound that increases with $K$ and tends to $\log p(x)$. The cell below computes it on the 20 demo images with the
decoder of the 2-D VAE, splitting 1,000 samples into groups of $K$ = 1, 10, 100, 1,000.

In [ ]:
ladder = []
for i in range(20):
    x = Xte_bin[SHOW[i]].astype(np.float64)
    eps = np.random.default_rng(500 + i).standard_normal((1000, 2))
    z = mu20[i] + sd20[i] * eps
    f = loglik_grad(dec["vae"], z, x)[0]
    log_w = (
        f - 0.5 * (z**2).sum(1) + 0.5 * (eps**2).sum(1) + np.log(sd20[i]).sum()
    )  # log p(x|z) + log p(z) − log q(z|x)
    kl = 0.5 * (mu20[i] ** 2 + sd20[i] ** 2 - 1 - np.log(sd20[i] ** 2)).sum()
    row = [f.mean() - kl]  # ELBO, with the closed-form KL
    for K in (1, 10, 100, 1000):
        g = log_w.reshape(-1, K)
        mx = g.max(1, keepdims=True)
        row.append(float((mx[:, 0] + np.log(np.exp(g - mx).mean(1))).mean()))
    ladder.append(row)
ladder = np.array(ladder).mean(0)
print("20 demo images, mean (nats): ELBO " + num(ladder[0], 2) + " · " + " · ".join(
    f"IWAE K = {num(K, 0)} {num(v, 2)}" for K, v in zip((1, 10, 100, 1000), ladder[1:])))  # fmt: skip
assert np.all(np.diff(ladder[1:]) > 0)  # nested groups: the bound doesn't decrease as K grows (Jensen's inequality)
print(f"IWAE K = 1,000 is {num(ladder[-1] - ladder[0], 2)} nats above the ELBO")
for n, label in [("vae2", "d = 2"), ("b1", "d = 50")]:
    s = G[n]["seeds"]
    print(f"whole test set, VAE {label} (demo, 3 seeds): −ELBO {span([-r['elbo'] for r in s], 1)} nats, "
          f"−IWAE₁₀₀₀ {span([-r['iwae'] for r in s], 1)} nats")  # fmt: skip

IWAE with $K = 1$ is the ELBO itself, except that the KL is estimated from samples instead of in closed form, so the
two numbers nearly coincide. Each increase in $K$ raises the bound; with $K = 1{,}000$, it is 2.56 nats above the
ELBO on these 20 images. On the whole test set, the gap of the 50-dimensional VAE is larger: −ELBO 95.5–95.7 nats,
while −IWAE-1,000 is 89.7–90.1. So maximizing the ELBO is not exactly the same as maximizing $\log p(x)$: the
model is also rewarded for making the true posterior easy to approximate with a Gaussian cloud.

## 11 · β: how many nats the code carries, and when a VAE collapses

The β-VAE multiplies the KL by β. $D_{\mathrm{KL}}$ is the price, in nats, that the code pays to carry information
about the image (averaged over images, it equals that very amount of information plus the distance from $q(z)$ to
$\mathcal N(0, I)$, section 8): the larger β, the more expensive information becomes. The demo trained
50-dimensional VAEs with β from 0.03 to 30 (3 seeds per β). The cell below prints, for each β: the KL and the
reconstruction error (nats on the test images), the number of **active dimensions** (dimensions $u$ with
$\mathrm{Var}_x\, \mu_u > 0.01$, the definition of Burda et al., 2016), the SSE on the grayscale images, and the FD
of samples drawn from $\mathcal N(0, I)$.

In [ ]:
BETAS = lat["setup"]["betas"]


def fb(b):
    """β as in the demo: 0.03 · 0.3 · 1 · 30."""
    return num(b, 2 if b < 0.1 else 1 if b < 1 else 0)


BURDA_ACTIVE = 19  # Burda, Grosse & Salakhutdinov (2016), Table 1: VAE with one 50-dimensional stochastic layer, k = 1


def beta_rows(b):
    return G[f"b{b:g}"]["seeds"]


print(f"{'β':>5} {'KL':>13} {'reconstruction':>14} {'active':>7} {'SSE':>13} {'FD N(0, I)':>15} {'FD GMM':>13}")
for b in BETAS:
    s = beta_rows(b)
    print(f"{fb(b):>5} {span([r['kl'] for r in s], 1):>13} "
          f"{span([r['rec'] for r in s], 1):>14} {span([r['active'] for r in s], 0):>7} {span([r['sse'] for r in s], 1):>13} "
          f"{span([r['prior']['fd'] for r in s], 0):>15} {span([r['gmm']['fd'] for r in s], 0):>13}")  # fmt: skip
small = [r for b in (0.03, 0.1) for r in beta_rows(b)]
rec1, sse1 = (np.median([r[k] for r in beta_rows(1)]) for k in ("rec", "sse"))
assert all(0.4 < r["rec"] / rec1 < 0.6 and abs(r["sse"] - sse1) < 0.5 for r in small)  # half the nats, same SSE

The figure below draws the same table on the **rate–distortion** plane (Alemi et al., 2018): the horizontal axis is
the KL (the *rate*: an upper bound on the number of nats the code carries about the image), the vertical axis is the
reconstruction error; each dot is one seed, labeled with its β. The dashed gray lines are the points with the same
−ELBO = reconstruction + KL. β slides along a trade-off curve; β = 1 is the point that touches the lowest −ELBO line.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
for c in (100, 150, 200):
    k = np.linspace(0, c, 2)
    ax.plot(k, c - k, color=GREY, ls="--", lw=0.8)
    ax.text(c * 0.98, 2, f"−ELBO = {c}", color=GREY, fontsize=8, ha="right")
for b in BETAS:
    s = beta_rows(b)
    ax.scatter([r["kl"] for r in s], [r["rec"] for r in s], s=30, color=BLUE, zorder=3)
    ax.annotate(f"β = {fb(b)}", (s[0]["kl"], s[0]["rec"]), xytext=(6, 4),
                textcoords="offset points", fontsize=9)  # fmt: skip
ax.set(xlabel="KL (nats): rate", ylabel="−log p(x|z): reconstruction nats (distortion)", xlim=(-5, 205), ylim=(0, 215))
ax.set_title("50-dimensional VAE by β (3 seeds): fewer nats in the code, worse reconstruction")
fmt_axes(fig)
plt.show()
elbo = {b: [-r["elbo"] for r in beta_rows(b)] for b in BETAS}
print(
    "−ELBO (nats, with β = 1 in the formula, so that it can be compared across β): "
    + " · ".join(f"β = {fb(b)}: {span(v, 1)}" for b, v in elbo.items())
)
print(f"Burda et al. (2016), 50-dimensional VAE on MNIST: {BURDA_ACTIVE}/50 active dimensions")

Reading the table from top to bottom:
- **Small β** (0.03–0.1): the KL is over 100 nats, all 50 dimensions are used, and the reconstruction error is only
  35.5–37.7 nats, about half that of β = 1 (the SSE on the grayscale images is level with β = 1). But samples from
  $\mathcal N(0, I)$ are very bad (FD above 100): the KL is too cheap, the clouds
  $q(z \mid x)$ don't need to overlap, and $\mathcal N(0, I)$ falls into the gaps between them, as with the
  autoencoder. A GMM still gives an FD of 19–23.
- **β = 1** (the original VAE): the lowest −ELBO in the table, and only **19–20 of the 50 dimensions** are active.
  Burda et al. (2016) also counted 19/50 on MNIST. The other dimensions have KL ≈ 0: the encoder gives them exactly
  $\mathcal N(0, 1)$ for every image, and the decoder ignores them.
- **Large β**: the KL is only a few nats, and the images get much blurrier (SSE 30–34 at β = 10). With β = 30,
  KL = 0 and no dimension is active: every image has the same code, and the decoder just draws a mean image. The SSE
  of 52.9 is exactly the error of “no compression, only the mean image $m$” in section 3 (52.89). This is
  **posterior collapse**: $q(z \mid x) = p(z)$ for every $x$.

Samples from $\mathcal N(0, I)$ are best at β = 1–3 (within the variation across seeds): enough KL for the clouds to
cover the origin completely, without losing too much information yet. β > 1 was once thought to help each code
dimension correspond to a separate factor that generates the image (β-VAE, Higgins et al., 2017); Locatello et al.
(2019) showed that without inductive biases or labels this cannot be guaranteed. This table only measures the
trade-off between the number of nats and the error.

### The β hidden in the Gaussian decoder
Many tutorials train VAEs on grayscale images with “MSE + KL”. A Gaussian decoder with a fixed variance $\sigma^2$
gives a reconstruction error of $\lVert \hat x - x\rVert^2 / (2\sigma^2)$ (plus a constant). Multiply the whole
loss by $2\sigma^2$:

$$\frac{\lVert \hat x - x\rVert^2}{2\sigma^2} + D_{\mathrm{KL}} \;\propto\; \lVert \hat x - x\rVert^2 + 2\sigma^2 D_{\mathrm{KL}}$$

Choosing $\sigma^2$ means choosing an implicit β, $\beta = 2\sigma^2$, relative to the sum of squared errors
(Doersch, 2016). “Sum of squares + KL” is $\sigma^2 = \tfrac12$. “MSE averaged over the 784 pixels + KL” is
$\lVert \hat x - x\rVert^2/784 + D_{\mathrm{KL}}$, i.e. $2\sigma^2 = 784$, $\sigma^2 = 392$: the KL weighs 784
times as much.

### 🤔 Predict first
We train two 8-dimensional VAEs on grayscale images that differ only in $\sigma^2$: ½ and 392. With
$\sigma^2 = 392$, how many of the 8 dimensions does the model use?

In [ ]:
gauss = {}
for s2 in (0.5, 392.0):
    p_g, _ = train(8, True, steps=600, batch=128, rec="gauss", s2=s2)
    gauss[s2] = evaluate(p_g, rec="gauss", s2=s2)
    r = gauss[s2]
    print(f"σ² = {num(s2, 1 if s2 < 1 else 0):>3}: KL {num(r['kl'], 3)} nats · {r['active']}/8 active dimensions · "
          f"SSE {num(r['sse'], 1)} (mean image only: {num(((Xte - m_pca) ** 2).sum(1).mean(), 1)})")  # fmt: skip
assert gauss[392.0]["active"] == 0 and gauss[392.0]["kl"] < 0.01 and gauss[0.5]["active"] == 8
print(f"largest variance of the images along one direction (largest PCA eigenvalue): λ₁ = {num(lam[0])}, "
      f"{num(392 / lam[0], 0)} times smaller than 392; {int((lam > 0.5).sum())} directions have a variance above ½")  # fmt: skip
for n in ("g0.5", "g392"):
    s = G[n]["seeds"]
    print(f"demo, 50-dimensional VAE, 20,000 steps, σ² = {n[1:]}: KL {span([abs(r['kl']) for r in s], 2)} nats, "
          f"{span([r['active'] for r in s], 0)}/50 dimensions, SSE {span([r['sse'] for r in s], 1)}")  # fmt: skip

None. With $\sigma^2 = 392$, the decoder assumes that each pixel has noise with a standard deviation of about 20,
while the pixels lie only in $[0, 1]$: every detail of the image is “just noise”, so paying KL for it is a losing
deal. The model chooses KL = 0, like β = 30 above. This short run hasn't even had time to draw the mean image exactly
(its SSE is larger than 52.9); the demo's 20,000-step runs collapse completely and reach exactly the SSE of the mean
image. For a linear VAE the solution can be computed exactly: it resembles probabilistic PCA (Tipping & Bishop,
1999; Lucas et al., 2019), and a direction is kept only when the variance of the data along it is larger than
$\sigma^2$. Here the largest variance is only 5.12, about 77 times smaller than 392. With $\sigma^2 = \tfrac12$, 23
directions have a variance above ½, and the VAE uses all 8 dimensions.

So VAE images are blurry for several overlapping reasons, and the KL is only one of them:
- a narrow bottleneck (section 6: the 2-D autoencoder and VAE are equally blurry);
- the picture we look at is the mean $\mathbb E[x \mid z]$, not a sample (section 7);
- the decoder's distribution: a large $\sigma^2$, or a large β, makes the code carry little information.

## 12 · Stable Diffusion: an autoencoder to compress, diffusion to generate

Stable Diffusion (Rombach et al., 2022) uses exactly the ideas of this lesson, but divides the work differently. Its
compressor is an autoencoder with a KL term, which turns a 512×512×3 image into a 64×64×4 latent grid. CompVis's
public configuration `autoencoder_kl_32x32x4` sets the KL weight `kl_weight` = 10⁻⁶, alongside an L1 loss, LPIPS
and a discriminator (GAN, Lesson 20). A KL that small only keeps the codes from drifting far away; it doesn't make
the codes shaped like $\mathcal N(0, I)$. The codes have a standard deviation of about 5.5, so Stable Diffusion
multiplies them by `scale_factor` = 0.18215 before feeding them to the diffusion model, and it never samples $z$ from
$\mathcal N(0, I)$ and decodes it: learning the distribution of the codes is the diffusion model's job (Lesson 21).
That is exactly the lesson of section 8: a good autoencoder + a separate density model for the codes.

In [ ]:
H, W, C_img = 512, 512, 3
f, c_lat = 8, 4  # 8× compression along each spatial dimension, 4 latent channels (SD 1.x; SD3 and FLUX use 16 channels)
n_img, n_lat = H * W * C_img, (H // f) * (W // f) * c_lat
print(
    f"image {H}×{W}×{C_img} = {num(n_img, 0)} numbers → code {H // f}×{W // f}×{c_lat} = {num(n_lat, 0)} numbers: {num(n_img / n_lat, 0)} times fewer"
)
print(
    f"16 channels: {num((H // f) * (W // f) * 16, 0)} numbers, {num(n_img / ((H // f) * (W // f) * 16), 0)} times fewer"
)
print(f"MNIST in section 6: 784 → 2 numbers, {num(784 / 2, 0)} times fewer · scale_factor 0.18215 = 1 / {num(1 / 0.18215, 2)} "
      f"(≈ 1 / {num(1 / 0.18215, 1)})")  # fmt: skip
assert n_img / n_lat == 48

## 13 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### How many components for the Gaussian mixture?
A GMM with $K = 1$ component is exactly the fitted Gaussian of section 8. The cell below fits GMMs with $K$ = 1, 3,
10, 30 components to the autoencoder's codes, then measures as in section 8: the share of samples that fall into a
gap, and the rarest digit. Change `KS_GMM` to try more.

In [ ]:
KS_GMM = (1, 3, 10, 30)
ref, held = codes["ae"][:4000], codes["ae"][4000:]
r99 = np.quantile(nearest(held, ref)[0], 0.99)
gmm_live = {}
for K in KS_GMM:
    z = gmm_sample(gmm_fit(ref, K=K), 2000, np.random.default_rng(7))
    dist, j = nearest(z, ref)
    share = np.bincount(code_lab[j], minlength=10) / len(z)
    gmm_live[K] = (dist > r99).mean()
    print(
        f"K = {K:>2}: {pct(gmm_live[K], 1):>5} of the samples fall into a gap; rarest digit ({share.argmin()}) makes up {pct(share.min(), 1)}"
    )
assert gmm_live[1] > 2 * gmm_live[10]

More components make the samples follow the clusters of codes more closely. But “falling into a gap” is only half
the story: a GMM with very many components, each hugging a single code, would only copy the images it has already
seen. Lesson 20 measures this copying by the distance to the nearest training image.

### β on the 2-D VAE
The cell below retrains the 2-D VAE of section 6 with β = 0.1, 1 and 10 (300 steps each, fewer than in section 6),
then prints the KL, the standard deviation of the codes and the SSE.

In [ ]:
beta_live = {}
for beta in (0.1, 1.0, 10.0):
    r = beta_live[beta] = evaluate(train(2, True, steps=300, batch=128, beta=beta)[0])
    print(f"β = {fb(beta):>3}: KL {num(r['kl'], 2)} nats · standard deviation of the codes ({num(r['mu'][:, 0].std())}, "
          f"{num(r['mu'][:, 1].std())}) · SSE {num(r['sse'])}")  # fmt: skip
assert beta_live[0.1]["kl"] > beta_live[1.0]["kl"] > beta_live[10.0]["kl"]
assert beta_live[0.1]["mu"].std() > beta_live[10.0]["mu"].std()
sses = [r["sse"] for r in beta_live.values()]
assert max(sses) / min(sses) < 1.1

The KL falls as β grows, and the codes shrink toward the origin. The SSE of this short run barely changes: in 2
dimensions the bottleneck is still the main limit, as in section 6. Only with 50 dimensions (section 11) does β
really change the reconstruction quality. Raise β to 30 or 100 and run again to watch first one, then both
dimensions switch off entirely.

## 14 · Limits and next steps

- **The samples are still blurry.** The Bernoulli decoder treats the pixels as independent given $z$; their mean is
  blurry, and the samples are speckled. Autoregressive decoders (PixelVAE) or hierarchical ones (NVAE) fix this
  part, but are much slower.
- **$\mathcal N(0, I)$ doesn't match the distribution of the codes.** Section 8 showed that a GMM fitted after
  training gives better samples; Stable Diffusion hands this part over to a diffusion model.
- **The measure is the ELBO, not image quality.** Maximizing the ELBO rewards explaining every pixel, noise
  included, not looking real.

Another direction drops the likelihood altogether: train the image generator using only a second network that judges
“real or fake”. That is the GAN, the next lesson.

## Summary
- The best linear compression is PCA: with k = 30, an error of 13.86 per image (13.87 when fitted on 60,000 images,
  as Hinton & Salakhutdinov did). A linear autoencoder learns exactly that subspace (99.9% overlap) but not the axes.
- Nonlinear layers compress much better: the deep autoencoder gets down to 3.63–3.69 at k = 30.
- In 2 dimensions, the autoencoder and the VAE are equally blurry (SSE 27.34–27.65 and 27.26–27.71): the blur comes
  from the bottleneck, not from the KL.
- The autoencoder's codes have an arbitrary scale; drawing from $\mathcal N(0, I)$ almost completely misses the 1s.
  The VAE's codes gather around the origin with a standard deviation close to 1. But an autoencoder + GMM gives
  samples on par with a VAE + $\mathcal N(0, I)$ (Ghosh et al.).
- The reparameterization trick: the variance of one sample is 768–49,780 times smaller than REINFORCE's; both are
  unbiased.
- The ELBO is a lower bound: IWAE is a few nats tighter. β = 1 gives the best −ELBO and uses only 19–20/50
  dimensions; with a large β the VAE collapses.
- “MSE + KL” ($\sigma^2 = 392$, i.e. β = 784) collapses completely: KL = 0, and every image becomes the mean image.

## Review questions
1. Why does a linear autoencoder reach exactly PCA's error, yet the rows of $W_d$ are not the principal components?
2. A 2-D autoencoder and a 2-D VAE have the same SSE. So what did the KL “buy”?
3. Why does drawing $z$ from a Gaussian with exactly the mean and covariance of the autoencoder's codes still give
   bad samples?
4. Both gradient estimators are unbiased. Why does REINFORCE need thousands of times more samples?
5. With β = 30, the VAE's SSE is exactly 52.9. What is that number, and why?
6. A tutorial uses `loss = mse(x_hat, x) + kl`, with `mse` averaged over the 784 pixels. What happens, and how do you
   fix it?

<details><summary>Hints and answers</summary>

1. The error depends only on the subspace spanned by $W_d$: rotating the directions within the subspace (and rotating
   back in $W_e$) leaves $\hat x$ unchanged. Baldi & Hornik (1989) proved that the minimum is the PCA subspace, not
   the axes.
2. Where the codes lie: the VAE's codes gather around the origin at the scale of $\mathcal N(0, I)$, so drawing
   $z \sim \mathcal N(0, I)$ and decoding gives all 10 digits. The autoencoder's codes spread far out, and
   $\mathcal N(0, I)$ covers only a corner of them.
3. The codes lie in clusters and streaks, with gaps in between. The Gaussian covers the gaps too, where the decoder
   was never trained (4.5–7.1% of the samples fall there). A Gaussian mixture follows the clusters, so it does much
   better.
4. REINFORCE uses only the value $\log p(x \mid z)$, a large number multiplied by $\varepsilon / \sigma$ with a small
   $\sigma$; it ignores the derivative $\partial \log p / \partial z$ that backpropagation can compute. A baseline
   reduces the variance, but it is still worse.
5. The error of the mean image $m$ (52.89 in section 3): with KL = 0 the code carries no information, and the best
   the decoder can do is draw the same mean image for every image. That is posterior collapse.
6. It is equivalent to $\sigma^2 = 392$, i.e. β = 784 relative to the sum of squares: the KL is so expensive that the
   VAE collapses. Sum the error over the pixels instead of averaging it (σ² = ½), or use a Bernoulli decoder on
   binary images, or learn $\sigma^2$.
</details>

## Further reading
- D. Kingma & M. Welling (2013), [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), and
  [An Introduction to Variational Autoencoders](https://arxiv.org/abs/1906.02691) (2019).
- D. Rezende, S. Mohamed & D. Wierstra (2014), [Stochastic Backpropagation](https://arxiv.org/abs/1401.4082).
- G. Hinton & R. Salakhutdinov (2006), [Reducing the Dimensionality of Data with Neural Networks](https://www.cs.toronto.edu/~hinton/absps/science.pdf).
- P. Baldi & K. Hornik (1989), [Neural networks and principal component analysis](https://doi.org/10.1016/0893-6080(89)90014-2).
- C. Doersch (2016), [Tutorial on Variational Autoencoders](https://arxiv.org/abs/1606.05908): section 2.4.3 on σ as a
  regularization parameter.
- Y. Burda, R. Grosse & R. Salakhutdinov (2016), [Importance Weighted Autoencoders](https://arxiv.org/abs/1509.00519):
  the IWAE bound and active dimensions.
- M. Hoffman & M. Johnson (2016), [ELBO surgery](https://approximateinference.org/2016/accepted/HoffmanJohnson2016.pdf):
  mean KL = the code's information + the distance from the aggregate distribution to the prior.
- A. Alemi et al. (2018), [Fixing a Broken ELBO](https://arxiv.org/abs/1711.00464): the rate–distortion plane.
- J. Lucas et al. (2019), [Don't Blame the ELBO!](https://arxiv.org/abs/1911.02469): posterior collapse in linear VAEs.
- B. Dai & D. Wipf (2019), [Diagnosing and Enhancing VAE Models](https://arxiv.org/abs/1903.05789).
- P. Ghosh et al. (2020), [From Variational to Deterministic Autoencoders](https://arxiv.org/abs/1903.12436).
- I. Higgins et al. (2017), [β-VAE](https://openreview.net/forum?id=Sy2fzU9gl); F. Locatello et al. (2019),
  [Challenging Common Assumptions in the Unsupervised Learning of Disentangled Representations](https://arxiv.org/abs/1811.12359).
- R. Rombach et al. (2022), [Latent Diffusion](https://arxiv.org/abs/2112.10752), and
  [the KL-f8 autoencoder configuration](https://github.com/CompVis/stable-diffusion/blob/main/configs/autoencoder/autoencoder_kl_32x32x4.yaml).
- [VAE Explainer](https://xnought.github.io/vae-explainer/) (Bertucci & Endert, 2024): a 2-D VAE running in the browser.

**Next lesson:** [Lesson 20 · GANs — adversarial networks](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gan/):
drop the likelihood, and learn to generate images with a second network that catches fakes.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")